# Module 2 · Assignment 2 — Multi-Omics Target Discovery: Parkinson's Disease

**Workflow:** load → harmonize gene symbols → concordance → multi-evidence score → rank → export

| Layer | Dataset | Source | Access |
|---|---|---|---|
| Genomics | GWAS Catalog, Parkinson disease (MONDO_0005180) | https://www.ebi.ac.uk/gwas | Open |
| Transcriptomics | GSE7621, substantia nigra, PD vs control (GEO2R) | https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE7621 | Open |
| Proteomics | Licker et al. 2014, *Proteomics* (PXD000427), substantia nigra | Supplementary Table (pmic7662-sup-0002) | Open |

Integration level: **gene-level** (layers come from different cohorts — unmatched).

## 0. Setup

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

RAW = Path("data/raw")
OUT = Path("data")

GWAS_FILE = RAW / "pd_gwas_association.tsv"
RNA_FILE  = RAW / "gse7621_geo2r.tsv"
PROT_FILE = RAW / "licker2014_supp.csv"

for f in [GWAS_FILE, RNA_FILE, PROT_FILE]:
    print(f"{f}: {'FOUND' if f.exists() else 'MISSING'}")

data\raw\pd_gwas_association.tsv: FOUND
data\raw\gse7621_geo2r.tsv: FOUND
data\raw\licker2014_supp.csv: FOUND


In [2]:
def pick_col(df, candidates, required=True):
    """Return the first column name in df matching one of the candidates (case-insensitive)."""
    lower = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    if required:
        raise KeyError(f"None of {candidates} found. Columns are: {list(df.columns)}")
    return None

def clean_symbol(s):
    return s.astype(str).str.strip().str.upper()

## 1. Genomics layer — GWAS Catalog

In [3]:
gw_raw = pd.read_csv(GWAS_FILE, sep="\t", low_memory=False)
print("Raw GWAS shape:", gw_raw.shape)
print(gw_raw.columns.tolist())

Raw GWAS shape: (905, 38)
['DATE ADDED TO CATALOG', 'PUBMEDID', 'FIRST AUTHOR', 'DATE', 'JOURNAL', 'LINK', 'STUDY', 'DISEASE/TRAIT', 'INITIAL SAMPLE SIZE', 'REPLICATION SAMPLE SIZE', 'REGION', 'CHR_ID', 'CHR_POS', 'REPORTED GENE(S)', 'MAPPED_GENE', 'UPSTREAM_GENE_ID', 'DOWNSTREAM_GENE_ID', 'SNP_GENE_IDS', 'UPSTREAM_GENE_DISTANCE', 'DOWNSTREAM_GENE_DISTANCE', 'STRONGEST SNP-RISK ALLELE', 'SNPS', 'MERGED', 'SNP_ID_CURRENT', 'CONTEXT', 'INTERGENIC', 'RISK ALLELE FREQUENCY', 'P-VALUE', 'PVALUE_MLOG', 'P-VALUE (TEXT)', 'OR or BETA', '95% CI (TEXT)', 'PLATFORM [SNPS PASSING QC]', 'CNV', 'MAPPED_TRAIT', 'MAPPED_TRAIT_URI', 'STUDY ACCESSION', 'GENOTYPING TECHNOLOGY']


In [4]:
trait_col = pick_col(gw_raw, ["MAPPED_TRAIT", "mappedTraits", "traitName", "DISEASE/TRAIT", "reportedTrait"], required=False)
if trait_col:
    print(f"Trait column: {trait_col}")
    print(gw_raw[trait_col].value_counts().head(20))

Trait column: MAPPED_TRAIT
MAPPED_TRAIT
Parkinson disease                                                                    698
drug-induced dyskinesia, response to levodopa                                         62
Parkinson disease, disease progression measurement                                    46
grey matter volume measurement, Parkinson disease                                     22
age at onset, Parkinson disease                                                       17
Lewy body attribute, Parkinson disease                                                10
survival time, Parkinson disease                                                      10
disease progression measurement                                                       10
trait in response to zonisamide, motor function measurement, response to levodopa      5
motor function measurement, response to levodopa                                       4
impulse control disorder                                              

In [5]:
gw = gw_raw.copy()
if trait_col:
    is_pd = gw[trait_col].astype(str).str.lower().str.strip() == "parkinson disease"
    if is_pd.sum() > 0:
        removed = gw.loc[~is_pd, trait_col].value_counts()
        gw = gw[is_pd].copy()
        print(f"Kept {len(gw)} associations; removed {int(removed.sum())} from other traits:")
        print(removed)
    else:
        print("No exact 'parkinson disease' label found - inspect value_counts above and adjust the filter.")
print("Associations after filter:", len(gw))

Kept 698 associations; removed 207 from other traits:
MAPPED_TRAIT
drug-induced dyskinesia, response to levodopa                                        62
Parkinson disease, disease progression measurement                                   46
grey matter volume measurement, Parkinson disease                                    22
age at onset, Parkinson disease                                                      17
Lewy body attribute, Parkinson disease                                               10
survival time, Parkinson disease                                                     10
disease progression measurement                                                      10
trait in response to zonisamide, motor function measurement, response to levodopa     5
motor function measurement, response to levodopa                                      4
impulse control disorder                                                              4
dementia, Parkinson disease, disease progression meas

In [6]:
gene_col = pick_col(gw, ["MAPPED_GENE", "mappedGenes", "REPORTED GENE(S)", "reportedGenes"])
mlog_col = pick_col(gw, ["PVALUE_MLOG"], required=False)
p_col    = pick_col(gw, ["P-VALUE", "pValue", "p-value", "PVALUE"], required=False)
print("Gene column:", gene_col, "| -log10 p column:", mlog_col, "| p column:", p_col)

def parse_p(x):
    """Handle numeric p-values and strings like '3 x 10-8' or '3E-8'."""
    if pd.isna(x):
        return np.nan
    s = str(x).replace("\u00d7", "x").replace(" ", "")
    m = re.match(r"^([\d.]+)x10\^?(-?\d+)$", s)
    if m:
        return float(m.group(1)) * 10 ** int(m.group(2))
    try:
        return float(s)
    except ValueError:
        return np.nan

if mlog_col:
    gw["neglog10p"] = pd.to_numeric(gw[mlog_col], errors="coerce")
else:
    gw["neglog10p"] = -np.log10(gw[p_col].map(parse_p))

gw["gene"] = gw[gene_col].astype(str).str.split(r"\s*[-,;]\s*")
gw = gw.explode("gene")
gw["gene"] = clean_symbol(gw["gene"])
gw = gw[~gw["gene"].isin(["", "NAN", "NA", "NONE"])]
gw = gw.dropna(subset=["neglog10p"])

gwas = (gw.groupby("gene", as_index=False)["neglog10p"].max()
          .sort_values("neglog10p", ascending=False))
gwas.to_csv(OUT / "pd_gwas.tsv", sep="\t", index=False)
print("GWAS gene-level table:", gwas.shape)
gwas.head(15)

Gene column: MAPPED_GENE | -log10 p column: PVALUE_MLOG | p column: P-VALUE
GWAS gene-level table: (393, 2)


,gene,neglog10p
327,SNCA,169.397940
195,LRRK2,147.397940
115,GBA1,89.522879
136,HMGN2P18,75.000000
355,TMEM175,74.301030
160,KRTCAP2,69.698970
179,LINC02210,69.000000
68,CRHR1,69.000000
201,MAPT,68.000000
22,ASH1L,67.397940


In [7]:
known_pd_genes = ["SNCA", "LRRK2", "GBA1", "GBA", "MAPT", "TMEM175", "GAK", "VPS13C", "BST1", "RAB29"]
print("Sanity check - known PD risk genes in GWAS layer:")
gwas[gwas["gene"].isin(known_pd_genes)]

Sanity check - known PD risk genes in GWAS layer:


,gene,neglog10p
327,SNCA,169.397940
195,LRRK2,147.397940
115,GBA1,89.522879
355,TMEM175,74.301030
201,MAPT,68.000000
36,BST1,32.221849
270,RAB29,29.000000
112,GAK,20.698970


## 2. Transcriptomics layer — GSE7621 (GEO2R)

In [8]:
r_raw = pd.read_csv(RNA_FILE, sep="\t")
print("Raw GEO2R shape (probes):", r_raw.shape)
print(r_raw.columns.tolist())

Raw GEO2R shape (probes): (54318, 8)
['ID', 'adj.P.Val', 'P.Value', 't', 'B', 'logFC', 'Gene.symbol', 'Gene.title']


In [9]:
sym_col = pick_col(r_raw, ["Gene.symbol", "Gene symbol", "GENE_SYMBOL", "Symbol"])
fc_col  = pick_col(r_raw, ["logFC", "log2FC"])
pv_col  = pick_col(r_raw, ["P.Value", "pvalue", "P.value"])

r = r_raw.dropna(subset=[sym_col]).copy()
n_multi = r[sym_col].astype(str).str.contains("///").sum()
r = r[~r[sym_col].astype(str).str.contains("///")]          # drop ambiguous multi-gene probes
r["gene"] = clean_symbol(r[sym_col])
r = r.sort_values(pv_col).drop_duplicates("gene")          # most significant probe per gene

rna = r[["gene", fc_col, pv_col]].rename(columns={fc_col: "log2fc", pv_col: "pval"})
print(f"Dropped {n_multi} multi-gene probes; {len(rna)} genes remain.")

Dropped 2197 multi-gene probes; 20799 genes remain.


**Direction check.** Dopaminergic markers (TH, SLC6A3, DDC) must be *down* in PD substantia nigra.
If they come out positive, the GEO2R contrast was control vs PD, so the sign is flipped to PD vs control.

In [10]:
markers = ["TH", "SLC6A3", "DDC"]
print("Before check:")
print(rna[rna["gene"].isin(markers)])

marker_median = rna.loc[rna["gene"].isin(markers), "log2fc"].median()
FLIPPED = bool(marker_median > 0)
if FLIPPED:
    rna["log2fc"] = -rna["log2fc"]
    print(f"\nMarker median log2FC was {marker_median:.2f} (positive) -> sign flipped to PD vs control.")
else:
    print(f"\nMarker median log2FC is {marker_median:.2f} (negative) -> contrast already PD vs control.")

print(rna[rna["gene"].isin(markers)])
rna.to_csv(OUT / "pd_rna.tsv", sep="\t", index=False)
print("RNA gene-level table:", rna.shape)

Before check:
        gene  log2fc      pval
1458  SLC6A3    2.15  0.017752
7218     DDC    1.94  0.135214
8298      TH    2.96  0.156665

Marker median log2FC was 2.15 (positive) -> sign flipped to PD vs control.
        gene  log2fc      pval
1458  SLC6A3   -2.15  0.017752
7218     DDC   -1.94  0.135214
8298      TH   -2.96  0.156665
RNA gene-level table: (20799, 3)


## 3. Proteomics layer — Licker et al. 2014 (PXD000427)
The supplement reports a PD/control ratio and a significance flag (`*`) per protein, but **no numeric p-values**.
The score below uses |log2FC| magnitude, so p-values are not required; the `significant` flag is kept for interpretation.

In [11]:
p = pd.read_csv(PROT_FILE)
print("Raw proteomics shape:", p.shape)
p["gene"] = clean_symbol(p["gene"])
p = p[p["gene"] != "NAN"]
prot = (p.sort_values("log2fc", key=abs, ascending=False)
          .drop_duplicates("gene")[["gene", "log2fc", "significant"]])
prot.to_csv(OUT / "pd_protein.tsv", sep="\t", index=False)
print("Protein gene-level table:", prot.shape)
print("Significant proteins:", int(prot["significant"].sum()))
prot[prot["gene"].isin(["TH", "FTL", "NEBL", "GFAP", "GGH", "BSCL2"])]

Raw proteomics shape: (1783, 5)
Protein gene-level table: (1779, 3)
Significant proteins: 204


,gene,log2fc,significant
1023,NEBL,0.8953,True
632,GFAP,0.7049,True
603,FTL,0.6508,True
634,GGH,-0.6439,True
1635,TH,-0.5778,False
229,BSCL2,-0.5778,True


## 4. Harmonize and join on gene symbol

In [12]:
layer_sizes = {"GWAS genes": len(gwas), "RNA genes": len(rna), "Protein genes": len(prot)}
for k, v in layer_sizes.items():
    print(f"{k}: {v}")

rna_prot = rna.merge(prot, on="gene", suffixes=("_rna", "_prot"))
all_three = rna_prot.merge(gwas, on="gene", how="inner")

print(f"\nRNA ∩ Protein: {len(rna_prot)}")
print(f"RNA ∩ Protein ∩ GWAS: {len(all_three)}")
print(f"RNA ∩ GWAS: {len(set(rna.gene) & set(gwas.gene))} | Protein ∩ GWAS: {len(set(prot.gene) & set(gwas.gene))}")

GWAS genes: 393
RNA genes: 20799
Protein genes: 1779

RNA ∩ Protein: 1684
RNA ∩ Protein ∩ GWAS: 34
RNA ∩ GWAS: 279 | Protein ∩ GWAS: 35


In [13]:
MIN_GENES = 30
USE_LEFT_JOIN = len(all_three) < MIN_GENES

if USE_LEFT_JOIN:
    merged = rna_prot.merge(gwas, on="gene", how="left")
    merged["in_gwas"] = merged["neglog10p"].notna()
    merged["neglog10p"] = merged["neglog10p"].fillna(0)
    print(f"Three-way overlap ({len(all_three)}) < {MIN_GENES}: using LEFT join for GWAS (missing = 0).")
else:
    merged = all_three.copy()
    merged["in_gwas"] = True
    print(f"Using INNER join across all three layers.")

print(f"Surviving genes: {len(merged)} ({int(merged['in_gwas'].sum())} with GWAS support)")

Using INNER join across all three layers.
Surviving genes: 34 (34 with GWAS support)


## 5. Concordance (RNA vs protein direction)

In [14]:
merged["concordant"] = np.sign(merged["log2fc_rna"]) == np.sign(merged["log2fc_prot"])
n_conc = int(merged["concordant"].sum())
print(f"Concordant: {n_conc} / {len(merged)} ({n_conc / len(merged):.1%})")

rho = merged[["log2fc_rna", "log2fc_prot"]].corr(method="spearman").iloc[0, 1]
print(f"Spearman correlation of RNA vs protein log2FC: {rho:.3f}")

sig = merged[merged["significant"]]
if len(sig):
    print(f"Among significant proteins: {int(sig['concordant'].sum())} / {len(sig)} concordant")

Concordant: 16 / 34 (47.1%)
Spearman correlation of RNA vs protein log2FC: -0.136
Among significant proteins: 2 / 6 concordant


## 6. Multi-evidence score (equal weights)

In [15]:
def minmax(s):
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng > 0 else s * 0

merged["rna_score"]  = minmax(merged["log2fc_rna"].abs())
merged["prot_score"] = minmax(merged["log2fc_prot"].abs())
merged["gwas_score"] = minmax(merged["neglog10p"])

EQUAL_WEIGHTS = {"rna_score": 1/3, "prot_score": 1/3, "gwas_score": 1/3}
merged["multi_evidence_score"] = sum(merged[k] * w for k, w in EQUAL_WEIGHTS.items())
print("Weights:", EQUAL_WEIGHTS)

Weights: {'rna_score': 0.3333333333333333, 'prot_score': 0.3333333333333333, 'gwas_score': 0.3333333333333333}


## 7. Rank and export

In [16]:
cols = ["gene", "log2fc_rna", "pval", "log2fc_prot", "significant", "neglog10p", "in_gwas",
        "concordant", "rna_score", "prot_score", "gwas_score", "multi_evidence_score"]
ranked = merged.sort_values("multi_evidence_score", ascending=False).reset_index(drop=True)[cols]
ranked.index = ranked.index + 1

TOP_N = 15
top = ranked.head(TOP_N)
top.to_csv("targets_parkinsons.csv", index_label="rank")
print(f"Exported top {TOP_N} targets to targets_parkinsons.csv")
top

Exported top 15 targets to targets_parkinsons.csv


,gene,log2fc_rna,pval,log2fc_prot,significant,neglog10p,in_gwas,concordant,rna_score,prot_score,gwas_score,multi_evidence_score
1,LMNA,3.650,0.001300,0.1635,True,20.698970,True,True,1.000000,0.459141,0.093299,0.517480
2,PBXIP1,-1.590,0.033304,0.2016,True,38.698970,True,False,0.430451,0.566133,0.203055,0.399880
3,SNCA,0.290,0.614742,-0.0439,False,169.397940,True,False,0.071028,0.123280,1.000000,0.398103
4,MAP4K4,-0.332,0.537204,0.3561,True,12.096910,True,False,0.082640,1.000000,0.040847,0.374496
5,WDR1,-0.564,0.317753,0.3219,True,6.096910,True,False,0.146783,0.903960,0.004262,0.351668
6,KTN1,-1.110,0.395551,-0.2009,False,9.698970,True,True,0.297741,0.564167,0.026226,0.296045
7,BAG3,-0.037,0.961725,0.2987,True,11.698970,True,False,0.001078,0.838809,0.038421,0.292770
8,NDUFAF2,0.739,0.206330,-0.2009,False,14.301030,True,False,0.195167,0.564167,0.054287,0.271207
9,MAPT,0.718,0.208475,0.0566,False,68.000000,True,True,0.189361,0.158944,0.381720,0.243342
10,PAM,0.416,0.476913,-0.2009,False,8.698970,True,False,0.105864,0.564167,0.020128,0.230053


## 8. Interpretation helpers 

In [17]:
check = ["SNCA", "LRRK2", "GBA1", "GBA", "MAPT", "PRKN", "PINK1", "PARK7", "TH", "SLC6A3", "ALDH1A1", "FTL", "GFAP"]
print("Known PD-relevant genes in the integrated table:")
ranked[ranked["gene"].isin(check)]

Known PD-relevant genes in the integrated table:


,gene,log2fc_rna,pval,log2fc_prot,significant,neglog10p,in_gwas,concordant,rna_score,prot_score,gwas_score,multi_evidence_score
3,SNCA,0.290,0.614742,-0.0439,False,169.39794,True,False,0.071028,0.123280,1.00000,0.398103
9,MAPT,0.718,0.208475,0.0566,False,68.00000,True,True,0.189361,0.158944,0.38172,0.243342


In [18]:
disc = ranked[~ranked["concordant"]].copy()
disc["abs_gap"] = (disc["log2fc_rna"] - disc["log2fc_prot"]).abs()
print("Most discordant genes (RNA and protein move in opposite directions):")
disc.sort_values("abs_gap", ascending=False).head(10)

Most discordant genes (RNA and protein move in opposite directions):


,gene,log2fc_rna,pval,log2fc_prot,significant,neglog10p,in_gwas,concordant,rna_score,prot_score,gwas_score,multi_evidence_score,abs_gap
2,PBXIP1,-1.590,0.033304,0.2016,True,38.698970,True,False,0.430451,0.566133,0.203055,0.399880,1.7916
16,TPM1,-1.690,0.034156,0.0286,False,5.698970,True,False,0.458099,0.080315,0.001836,0.180083,1.7186
14,FYN,-1.260,0.097902,0.0566,False,14.397940,True,False,0.339213,0.158944,0.054878,0.184345,1.3166
11,SH3GL2,1.220,0.403402,-0.0740,False,20.096910,True,False,0.328154,0.207807,0.089628,0.208530,1.2940
25,CAMK2D,0.984,0.185318,0.0000,False,12.000000,True,False,0.262905,0.000000,0.040256,0.101054,0.9840
8,NDUFAF2,0.739,0.206330,-0.2009,False,14.301030,True,False,0.195167,0.564167,0.054287,0.271207,0.9399
5,WDR1,-0.564,0.317753,0.3219,True,6.096910,True,False,0.146783,0.903960,0.004262,0.351668,0.8859
13,CHL1,-0.748,0.329414,0.1375,False,5.397940,True,False,0.197655,0.386127,0.000000,0.194594,0.8855
26,SCN2A,0.763,0.663012,-0.0291,False,7.522879,True,False,0.201803,0.081719,0.012957,0.098826,0.7921
4,MAP4K4,-0.332,0.537204,0.3561,True,12.096910,True,False,0.082640,1.000000,0.040847,0.374496,0.6881


In [19]:
summary = {
    "GWAS genes": len(gwas),
    "RNA genes": len(rna),
    "Protein genes": len(prot),
    "RNA ∩ Protein": len(rna_prot),
    "RNA ∩ Protein ∩ GWAS": len(all_three),
    "GWAS join": "left (missing = 0)" if USE_LEFT_JOIN else "inner",
    "RNA sign flipped": FLIPPED,
    "Concordant fraction": round(n_conc / len(merged), 3),
    "Spearman RNA vs protein": round(rho, 3),
}
pd.Series(summary, name="value").to_frame()

,value
GWAS genes,393
RNA genes,20799
Protein genes,1779
RNA ∩ Protein,1684
RNA ∩ Protein ∩ GWAS,34
GWAS join,inner
RNA sign flipped,True
Concordant fraction,0.471
Spearman RNA vs protein,-0.136
